# Change Blindness

> **Task name:** `Change Blindness`

**Track: Attention** | Can the model detect subtle changes across an interruption?

Humans often fail to notice changes in a scene when the change coincides with a
brief disruption (a blink, a flicker, a cut). This benchmark tests whether LLMs
can detect factual changes between two versions of a passage when separated by
a disruptor paragraph.

**Metrics:**
- Detection rate by change magnitude (minor vs. major)
- Detection rate by disruptor length (0, 1, 3 paragraphs)
- False alarm rate (reporting non-existent changes)
- Specificity (identifying exactly what changed)

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.lower().strip())

In [ ]:
# --- Passages with controlled changes ---
# Each passage has a "minor" change (a number/name) and a "major" change (a causal claim).
# Passages are short, everyday topics — easy to read and verify at a glance.

PASSAGES = [
    {
        "id": "bakery",
        "original": "Rosa's Bakery on Elm Street has been open since 1987. They bake 400 loaves of sourdough every morning using a 50-year-old starter culture. The shop employs 12 full-time staff and sources all its flour from a local mill 30 miles away. Customers line up before dawn because the bread sells out by 9 AM.",
        "minor_change": "Rosa's Bakery on Elm Street has been open since 1987. They bake 250 loaves of sourdough every morning using a 50-year-old starter culture. The shop employs 12 full-time staff and sources all its flour from a local mill 30 miles away. Customers line up before dawn because the bread sells out by 9 AM.",
        "minor_detail": "400 loaves changed to 250 loaves",
        "major_change": "Rosa's Bakery on Elm Street has been open since 1987. They bake 400 loaves of sourdough every morning using a 50-year-old starter culture. The shop employs 12 full-time staff and sources all its flour from a local mill 30 miles away. Customers rarely buy the bread because most of it gets thrown away unsold.",
        "major_detail": "bread sells out by 9 AM changed to most of it gets thrown away unsold",
    },
    {
        "id": "bridge",
        "original": "The Millbrook Bridge was built in 1924 and spans 840 feet across the Cooper River. Engineer James Whitfield designed it to carry 6 lanes of traffic. The bridge uses 14,000 tons of steel and was painted its signature green in 1962. Annual inspections confirm the structure remains safe for another 30 years.",
        "minor_change": "The Millbrook Bridge was built in 1924 and spans 840 feet across the Cooper River. Engineer James Whitfield designed it to carry 4 lanes of traffic. The bridge uses 14,000 tons of steel and was painted its signature green in 1962. Annual inspections confirm the structure remains safe for another 30 years.",
        "minor_detail": "6 lanes changed to 4 lanes",
        "major_change": "The Millbrook Bridge was built in 1924 and spans 840 feet across the Cooper River. Engineer James Whitfield designed it to carry 6 lanes of traffic. The bridge uses 14,000 tons of steel and was painted its signature green in 1962. Annual inspections reveal severe corrosion that could cause a collapse within 2 years.",
        "major_detail": "remains safe for another 30 years changed to severe corrosion that could cause a collapse within 2 years",
    },
    {
        "id": "farm",
        "original": "Sunrise Farm grows organic strawberries on 45 acres in the Salinas Valley. Owner Maria Chen switched to drip irrigation in 2018, cutting water use by 35 percent. The farm produces 800 crates per week during peak season and sells directly to 20 local restaurants. Last year's revenue was $1.2 million, a record for the farm.",
        "minor_change": "Sunrise Farm grows organic strawberries on 45 acres in the Salinas Valley. Owner Maria Chen switched to drip irrigation in 2018, cutting water use by 35 percent. The farm produces 800 crates per week during peak season and sells directly to 9 local restaurants. Last year's revenue was $1.2 million, a record for the farm.",
        "minor_detail": "20 local restaurants changed to 9 local restaurants",
        "major_change": "Sunrise Farm grows organic strawberries on 45 acres in the Salinas Valley. Owner Maria Chen switched to drip irrigation in 2018, cutting water use by 35 percent. The farm produces 800 crates per week during peak season and sells directly to 20 local restaurants. Last year's revenue fell to just $80,000 due to a devastating pest outbreak.",
        "major_detail": "revenue was $1.2 million a record changed to revenue fell to just $80,000 due to pest outbreak",
    },
    {
        "id": "library",
        "original": "The Greystone Public Library holds 120,000 books across three floors. Head librarian David Park introduced a free tutoring program in 2020 that now serves 300 students per month. The building was renovated in 2015 with a new reading garden funded by a $2 million donation. Visitor numbers have increased every year since the renovation.",
        "minor_change": "The Greystone Public Library holds 120,000 books across three floors. Head librarian David Park introduced a free tutoring program in 2020 that now serves 300 students per month. The building was renovated in 2015 with a new reading garden funded by a $500,000 donation. Visitor numbers have increased every year since the renovation.",
        "minor_detail": "$2 million donation changed to $500,000 donation",
        "major_change": "The Greystone Public Library holds 120,000 books across three floors. Head librarian David Park introduced a free tutoring program in 2020 that now serves 300 students per month. The building was renovated in 2015 with a new reading garden funded by a $2 million donation. Visitor numbers have dropped sharply every year and the library faces permanent closure.",
        "major_detail": "visitor numbers increased every year changed to dropped sharply and faces permanent closure",
    },
    {
        "id": "coach",
        "original": "Coach Diane Torres led the Westlake High swim team to 8 consecutive state titles. She trains 32 swimmers year-round using interval sets she developed during her Olympic career. The team practices at the county aquatic center, which has a 50-meter competition pool heated to 79 degrees. Her coaching philosophy emphasizes technique over raw endurance.",
        "minor_change": "Coach Diane Torres led the Westlake High swim team to 8 consecutive state titles. She trains 15 swimmers year-round using interval sets she developed during her Olympic career. The team practices at the county aquatic center, which has a 50-meter competition pool heated to 79 degrees. Her coaching philosophy emphasizes technique over raw endurance.",
        "minor_detail": "32 swimmers changed to 15 swimmers",
        "major_change": "Coach Diane Torres led the Westlake High swim team to 8 consecutive state titles. She trains 32 swimmers year-round using interval sets she developed during her Olympic career. The team practices at the county aquatic center, which has a 50-meter competition pool heated to 79 degrees. Her coaching philosophy emphasizes pure endurance training with no attention to technique.",
        "major_detail": "emphasizes technique over raw endurance changed to emphasizes pure endurance with no attention to technique",
    },
]

DISRUPTORS = [
    "The average American household throws away about 30 percent of the food it buys each year. Most of this waste comes from fresh produce that spoils before it can be eaten.",
    "Standard shipping containers are exactly 20 or 40 feet long. This uniform size lets them move between ships, trains, and trucks without being unpacked.",
    "The first commercial crossword puzzle book was published in 1924 and became an instant bestseller. Public libraries reported that puzzle fans were tearing pages out of dictionaries.",
]

CONFIDENCE_SUFFIX = (
    "\n\nAfter stating the change (or NO CHANGE), rate your confidence on a scale of 1-5:\n"
    "1 = Very unsure, 2 = Somewhat unsure, 3 = Neutral, 4 = Somewhat confident, 5 = Very confident\n\n"
    "Format:\n"
    "Change: <your answer>\n"
    "Confidence: <1-5>"
)

random.seed(123)
data = []
task_id = 0

for passage in PASSAGES:
    for change_type in ["minor", "major"]:
        changed_text = passage[f"{change_type}_change"]
        expected_detail = passage[f"{change_type}_detail"]

        for disruptor_count in [0, 1, 3]:
            disruptor_text = ""
            if disruptor_count > 0:
                selected = random.sample(DISRUPTORS, min(disruptor_count, len(DISRUPTORS)))
                if disruptor_count > len(DISRUPTORS):
                    selected = selected * (disruptor_count // len(DISRUPTORS) + 1)
                    selected = selected[:disruptor_count]
                disruptor_text = "\n\n".join(selected)

            if disruptor_count == 0:
                prompt = (
                    "Read Version A and Version B of the following passage carefully. "
                    "Identify what factual detail changed between them.\n\n"
                    f"=== VERSION A ===\n{passage['original']}\n\n"
                    f"=== VERSION B ===\n{changed_text}\n\n"
                    "What specific factual detail changed between Version A and Version B? "
                    "State the change precisely: what was the original detail and what did it become?"
                    + CONFIDENCE_SUFFIX
                )
            else:
                prompt = (
                    "Read Version A, then the interlude, then Version B carefully. "
                    "Identify what factual detail changed between the two versions.\n\n"
                    f"=== VERSION A ===\n{passage['original']}\n\n"
                    f"=== INTERLUDE ===\n{disruptor_text}\n\n"
                    f"=== VERSION B ===\n{changed_text}\n\n"
                    "What specific factual detail changed between Version A and Version B? "
                    "Ignore the interlude — it is unrelated. "
                    "State the change precisely: what was the original detail and what did it become?"
                    + CONFIDENCE_SUFFIX
                )

            data.append({
                "task_id": task_id,
                "prompt": prompt,
                "expected": expected_detail,
                "passage_id": passage["id"],
                "change_type": change_type,
                "disruptor_count": disruptor_count,
            })
            task_id += 1

    # Control: no-change condition (false alarm test)
    for disruptor_count in [0, 1, 3]:
        disruptor_text = ""
        if disruptor_count > 0:
            selected = random.sample(DISRUPTORS, min(disruptor_count, len(DISRUPTORS)))
            if disruptor_count > len(DISRUPTORS):
                selected = selected * (disruptor_count // len(DISRUPTORS) + 1)
                selected = selected[:disruptor_count]
            disruptor_text = "\n\n".join(selected)

        if disruptor_count == 0:
            prompt = (
                "Read Version A and Version B of the following passage carefully. "
                "Identify what factual detail changed between them. "
                "If nothing changed, respond with exactly: NO CHANGE\n\n"
                f"=== VERSION A ===\n{passage['original']}\n\n"
                f"=== VERSION B ===\n{passage['original']}\n\n"
                "What specific factual detail changed between Version A and Version B? "
                "If the passages are identical, respond with exactly: NO CHANGE"
                + CONFIDENCE_SUFFIX
            )
        else:
            prompt = (
                "Read Version A, then the interlude, then Version B carefully. "
                "Identify what factual detail changed between the two versions. "
                "If nothing changed, respond with exactly: NO CHANGE\n\n"
                f"=== VERSION A ===\n{passage['original']}\n\n"
                f"=== INTERLUDE ===\n{disruptor_text}\n\n"
                f"=== VERSION B ===\n{passage['original']}\n\n"
                "What specific factual detail changed between Version A and Version B? "
                "Ignore the interlude — it is unrelated. "
                "If the passages are identical, respond with exactly: NO CHANGE"
                + CONFIDENCE_SUFFIX
            )

        data.append({
            "task_id": task_id,
            "prompt": prompt,
            "expected": "NO CHANGE",
            "passage_id": passage["id"],
            "change_type": "none",
            "disruptor_count": disruptor_count,
        })
        task_id += 1

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} items")
print(f"By change type: {dict(df_all['change_type'].value_counts())}")
print(f"By disruptor count: {dict(df_all['disruptor_count'].value_counts())}")

## Task Definition

In [ ]:
@kbench.task(
    name="change_blindness",
    description="Detect subtle factual changes between two passage versions separated by a disruptor paragraph"
)
def change_blindness(
    llm,
    prompt: str,
    expected: str,
    change_type: str,
    disruptor_count: int,
) -> bool:
    response = llm.prompt(prompt)
    response = strip_thinking(response)
    resp_norm = normalize(response)
    exp_norm = normalize(expected)

    if expected == "NO CHANGE":
        # Correct if model says no change
        return "no change" in resp_norm

    # For actual changes: check if key details are mentioned
    # Split expected into original->changed parts
    key_terms = [normalize(t) for t in re.split(r"changed to|→|->", expected.lower()) if t.strip()]
    if len(key_terms) >= 2:
        # Model should mention both the old and new values
        return all(term in resp_norm for term in key_terms)
    else:
        # Fallback: substring match
        return exp_norm in resp_norm

## Run Evaluation

In [ ]:
eval_df = df_all[["prompt", "expected", "change_type", "disruptor_count"]].copy()

print(f"Running {len(eval_df)} items with kbench.llm...")
runs = change_blindness.evaluate(
    llm=kbench.llm,
    evaluation_data=eval_df,
    n_jobs=2,
    timeout=120,
    max_attempts=2,
)
results = runs.as_dataframe()
results = results.merge(
    df_all[["prompt", "passage_id", "change_type", "disruptor_count"]],
    on="prompt", how="left", suffixes=("", "_meta")
)
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
# Convert bool result to float
results["correct"] = results["result"].apply(lambda r: float(r) if isinstance(r, bool) else float(r))

print("=== Detection Rate by Change Type × Disruptor Count ===")
pivot = results.groupby(["change_type", "disruptor_count"])["correct"].mean().unstack()
print(pivot.to_string(float_format="%.2f"))

print("\n=== Overall by Change Type ===")
print(results.groupby("change_type")["correct"].agg(["mean", "count"]).to_string(float_format="%.2f"))

# False alarm rate
no_change = results[results["change_type"] == "none"]
false_alarm = 1 - no_change["correct"].mean() if len(no_change) > 0 else 0
print(f"\nFalse alarm rate (saying something changed when nothing did): {false_alarm:.2%}")

# Change detection drop with disruptors
for ct in ["minor", "major"]:
    ct_data = results[results["change_type"] == ct]
    baseline = ct_data[ct_data["disruptor_count"] == 0]["correct"].mean()
    with_3 = ct_data[ct_data["disruptor_count"] == 3]["correct"].mean()
    print(f"{ct} change: baseline={baseline:.2%}, with 3 disruptors={with_3:.2%}, drop={baseline-with_3:.2%}")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for ct, marker, color in [("minor", "s", "orange"), ("major", "^", "blue"), ("none", "o", "red")]:
    subset = results[results["change_type"] == ct]
    curve = subset.groupby("disruptor_count")["correct"].mean()
    label = f"{ct} change" if ct != "none" else "no change (1 - false alarm)"
    vals = curve.values if ct != "none" else 1 - curve.values
    ax.plot(curve.index, vals, f"{marker}-", label=label, linewidth=2, markersize=8, color=color)

ax.set_xlabel("Number of Disruptor Paragraphs")
ax.set_ylabel("Detection Rate")
ax.set_title("Change Blindness: Detection Rate by Disruption Level")
ax.set_xticks([0, 1, 3])
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig("change_blindness_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to change_blindness_results.png")

## Confidence Calibration Analysis

Does the model know when it's wrong? We parse the confidence ratings (1-5) from responses and compare against actual accuracy to measure metacognitive calibration.

In [ ]:
# Parse confidence from model responses
def extract_confidence(response_text: str) -> int | None:
    """Extract confidence rating (1-5) from response."""
    text = strip_thinking(str(response_text))
    # Look for "Confidence: N" pattern
    match = re.search(r"[Cc]onfidence\s*[:=]\s*(\d)", text)
    if match:
        val = int(match.group(1))
        return val if 1 <= val <= 5 else None
    return None

# Extract confidence from raw responses
if "response" in results.columns:
    results["confidence"] = results["response"].apply(extract_confidence)
else:
    # Try to get from the evaluation run
    results["confidence"] = None
    print("Note: Raw responses not available in results DataFrame.")
    print("Confidence analysis requires the raw response text.")

conf_results = results.dropna(subset=["confidence"]).copy()
conf_results["confidence"] = conf_results["confidence"].astype(int)

if len(conf_results) > 0:
    print(f"=== Confidence Calibration ({len(conf_results)} items with confidence) ===\n")
    
    # Calibration table: for each confidence level, what's the actual accuracy?
    print("Confidence | Accuracy | Count | Calibration Gap")
    print("-" * 55)
    
    calibration_data = []
    for conf_level in range(1, 6):
        subset = conf_results[conf_results["confidence"] == conf_level]
        if len(subset) > 0:
            acc = subset["correct"].mean()
            expected_acc = conf_level / 5.0  # Linear mapping: 1→0.2, 5→1.0
            gap = acc - expected_acc
            calibration_data.append({
                "confidence": conf_level,
                "accuracy": acc,
                "expected": expected_acc,
                "count": len(subset),
                "gap": gap,
            })
            print(f"    {conf_level}      |  {acc:.2%}  |  {len(subset):3d}  | {gap:+.2%}")
    
    cal_df = pd.DataFrame(calibration_data)
    
    # Expected Calibration Error (ECE)
    if len(cal_df) > 0:
        total = cal_df["count"].sum()
        ece = sum(row["count"] / total * abs(row["gap"]) for _, row in cal_df.iterrows())
        print(f"\nExpected Calibration Error (ECE): {ece:.4f}")
    
    # Overconfidence: wrong answers with confidence 4-5
    wrong_high_conf = conf_results[(conf_results["correct"] == 0) & (conf_results["confidence"] >= 4)]
    total_wrong = conf_results[conf_results["correct"] == 0]
    if len(total_wrong) > 0:
        overconf_rate = len(wrong_high_conf) / len(total_wrong)
        print(f"Overconfidence rate (wrong + conf>=4): {overconf_rate:.2%} ({len(wrong_high_conf)}/{len(total_wrong)})")
    
    # Underconfidence: correct answers with confidence 1-2
    correct_low_conf = conf_results[(conf_results["correct"] == 1) & (conf_results["confidence"] <= 2)]
    total_correct = conf_results[conf_results["correct"] == 1]
    if len(total_correct) > 0:
        underconf_rate = len(correct_low_conf) / len(total_correct)
        print(f"Underconfidence rate (correct + conf<=2): {underconf_rate:.2%} ({len(correct_low_conf)}/{len(total_correct)})")
    
    # Calibration by change type
    print("\n=== Confidence by Change Type ===")
    print(conf_results.groupby("change_type")[["confidence", "correct"]].mean().to_string(float_format="%.2f"))
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Reliability diagram
    if len(cal_df) > 0:
        ax1.bar(cal_df["confidence"], cal_df["accuracy"], width=0.6, color="steelblue", 
                edgecolor="black", linewidth=0.5, label="Actual accuracy")
        ax1.plot([1, 5], [0.2, 1.0], "r--", linewidth=2, label="Perfect calibration")
        ax1.set_xlabel("Confidence Level")
        ax1.set_ylabel("Actual Accuracy")
        ax1.set_title("Reliability Diagram")
        ax1.set_xticks([1, 2, 3, 4, 5])
        ax1.set_ylim(0, 1.1)
        ax1.legend()
    
    # Confidence distribution for correct vs incorrect
    for label, subset, color in [
        ("Correct", conf_results[conf_results["correct"] == 1], "seagreen"),
        ("Incorrect", conf_results[conf_results["correct"] == 0], "tomato"),
    ]:
        if len(subset) > 0:
            counts = subset["confidence"].value_counts().reindex(range(1, 6), fill_value=0)
            offset = 0.2 if label == "Correct" else -0.2
            ax2.bar(counts.index + offset, counts.values, width=0.35, color=color,
                    edgecolor="black", linewidth=0.5, label=label)
    
    ax2.set_xlabel("Confidence Level")
    ax2.set_ylabel("Count")
    ax2.set_title("Confidence Distribution: Correct vs Incorrect")
    ax2.set_xticks([1, 2, 3, 4, 5])
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig("change_blindness_calibration.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Calibration plot saved to change_blindness_calibration.png")
else:
    print("No confidence data available — model may not have followed the confidence format.")